In [0]:
from pyspark.sql.functions import * 

In [0]:
df_bronze_patient = spark.readStream.table('healthcare.bronze.patient')
silver_table = 'healthcare.silver.patient'
checkpoint_path = '/Volumes/healthcare/silver/checkpoint/silver_patient_files/v1' 

In [0]:
df_patient_clean = df_bronze_patient.dropDuplicates(['patient_id']).withColumn('updated_at', current_timestamp())

In [0]:
def merge_patient_data(batch_df,batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format('delta').mode('append').saveAsTable(silver_table)
        return 
    
    from delta.tables import DeltaTable

    final_table = DeltaTable.forName(spark,silver_table)

    (
        final_table.alias('Target')
            .merge(batch_df.alias('source'),
            'Target.patient_id = source.patient_id'
            )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


(
    df_patient_clean
        .writeStream
        .format('delta')
        .foreachBatch(merge_patient_data)
        .option('checkpointLocation',checkpoint_path)
        .trigger(availableNow = True)
        .start()
)